# EC2 L40S Smoke Test Runner

Run this notebook inside Jupyter on the EC2 machine with kernel `L40 PyTorch`.

It runs short from-scratch pretraining smoke tests for:

1. TinyStories + MiniGPT-Dense-60M-v1 + AdamW
2. TinyStories + MiniGPT-Dense-60M-v1 + Hybrid Muon


In [ ]:
import torch, os, subprocess, json, pathlib
print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('cwd:', os.getcwd())


## AdamW smoke test

This prepares a capped TinyStories token cache and trains for a few steps/minutes.


In [ ]:
!python gpu_benchmark/train_gpt.py \
  --run-id tinystories_60m_adamw_smoke_nb \
  --run-dir /tmp/tinystories_60m_adamw_smoke_nb \
  --dataset tinystories \
  --data-dir /tmp/tinystories_gpt2_512_smoke \
  --prepare-data \
  --max-train-tokens 1000000 \
  --max-val-tokens 131072 \
  --tokenizer openai-community/gpt2 \
  --model-config minigpt_dense_60m_v1 \
  --optimizer adamw \
  --precision bf16 \
  --micro-batch-size 2 \
  --grad-accum-steps 2 \
  --max-steps 3 \
  --max-minutes 0 \
  --eval-every-steps 1 \
  --save-every-steps 1 \
  --validation-tokens 16384 \
  --sample-new-tokens 20


## Muon smoke test

Uses the same tokenized TinyStories cache, but switches optimizer to Hybrid Muon.


In [ ]:
!python gpu_benchmark/train_gpt.py \
  --run-id tinystories_60m_muon_smoke_nb \
  --run-dir /tmp/tinystories_60m_muon_smoke_nb \
  --dataset tinystories \
  --data-dir /tmp/tinystories_gpt2_512_smoke \
  --tokenizer openai-community/gpt2 \
  --model-config minigpt_dense_60m_v1 \
  --optimizer muon \
  --precision bf16 \
  --micro-batch-size 2 \
  --grad-accum-steps 2 \
  --max-steps 3 \
  --max-minutes 0 \
  --eval-every-steps 1 \
  --save-every-steps 1 \
  --validation-tokens 16384 \
  --sample-new-tokens 20


## Inspect artifacts


In [ ]:
!find /tmp/tinystories_60m_adamw_smoke_nb /tmp/tinystories_60m_muon_smoke_nb -maxdepth 3 -type f | sort
